In [ ]:
# Sort data by user and order
order_products_seq = order_products.sort_values(by=['user_id', 'order_number'])

# Create purchase history per user
user_sequences = order_products_seq.groupby('user_id')['product_id'].apply(list)

# Filter users with enough data
user_sequences = user_sequences[user_sequences.apply(len) >= 5]

# Flatten into (X_seq, y) pairs
sequence_data = []
seq_len = 3  # you can tune this

for user, products in user_sequences.items():
    for i in range(seq_len, len(products)):
        X_seq = products[i-seq_len:i]
        y = products[i]
        sequence_data.append((X_seq, y))

print("Total sequences:", len(sequence_data))


Encode Product IDs

We need to convert product IDs into contiguous integer indices and later into one-hot or embedding format.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Flatten sequences and targets
X_seq_all, y_all = zip(*sequence_data)

# Fit encoder on all product IDs
product_encoder = LabelEncoder()
product_encoder.fit([pid for seq in X_seq_all for pid in seq] + list(y_all))

# Flatten all sequences into a single 1D list
flattened_X = [pid for seq in X_seq_all for pid in seq]
X_flat_encoded = product_encoder.transform(flattened_X)


# Reshape back into original (n_sequences, seq_len)
X_encoded = X_flat_encoded.reshape(-1, 3)

# Encode targets
y_encoded = product_encoder.transform(y_all)

vocab_size = len(product_encoder.classes_)
print("Unique products:", vocab_size)


In [ ]:
!pip install tensorflow

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import numpy as np

# Pad input sequences to fixed length
X_padded = pad_sequences(X_encoded, maxlen=3, padding='pre')  # already fixed-length, but standardizes shape

# Convert to NumPy
X_np = np.array(X_padded)
y_np = np.array(y_encoded)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

print("Train samples:", len(X_train), "Test samples:", len(X_test))


Training the LSTM Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=64, input_length=3))
model.add(LSTM(64, return_sequences=False))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])
model.summary()


In [ ]:
model.fit(
    X_train[:500_000], 
    y_train[:500_000], 
    epochs=3, 
    batch_size=512, 
    validation_split=0.1
)


In [ ]:
# Predict probabilities
y_probs = model.predict(X_test[:5])

# Get top predicted product for each sequence
y_pred_ids = np.argmax(y_probs, axis=1)

# Decode product names
predicted_products = product_encoder.inverse_transform(y_pred_ids)
true_products = product_encoder.inverse_transform(y_test[:5])

for i in range(5):
    print(f"Input Sequence: {product_encoder.inverse_transform(X_test[i])}")
    print(f"Predicted NBO: {predicted_products[i]}")
    print(f"Actual Next:    {true_products[i]}")
    print("—" * 40)


In [ ]:
import numpy as np

# Predict probabilities
y_pred_probs = model.predict(X_test[:1000])

# Top-K predictions manually
k = 5
top_k_preds = np.argsort(y_pred_probs, axis=1)[:, -k:]

# Count how often true label appears in top-k
y_true_subset = y_test[:1000]
correct = sum(y in preds for y, preds in zip(y_true_subset, top_k_preds))

top_k_accuracy = correct / len(y_true_subset)
print(f"Top-{k} Accuracy: {top_k_accuracy:.4f}")



